In [1]:
# Save global sum of mortality for all ensemble members

In [1]:
import os
import xarray as xr
from utils.utils import get_scenario_config

In [6]:
# === Path config ===
MORTALITY_DIR = "/glade/work/awells/air_quality/CESM/mortality/ozone/"

# Set to whatever scenario you want, function returns error if not recognised
scenario = "SSP245_G6"

config = get_scenario_config(scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

# === Main loop ===
ensembles = []
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")
    # {years.stop - 1} from OSDMA8 calculation
    dates = f"{years.start}-{years.stop - 1}"

    in_file = f"Mortality_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
    in_path = os.path.join(MORTALITY_DIR, in_file)

    if not os.path.exists(in_path):
        print(f"Missing: {in_path}")
        continue
    da = xr.open_dataarray(in_path)

    global_sum = da.sum(dim=("lat", "lon"))
    ensembles.append(global_sum)

ensemble_sum = xr.concat(ensembles,
                         dim=(xr.DataArray(ensemble_members,
                                           dims="ensemble", name="ensemble")))

ensemble_sum.attrs["description"] = ("Global mean Mortality - scripts by "
                                     "A.F. Wells (2025)")

out_file = f"Mortality_global_sum_CESM2_{scenario}_{dates}.nc"
out_path = os.path.join(MORTALITY_DIR, out_file)
print(f"Saving global sum in {out_path}")
ensemble_sum.to_netcdf(out_path)

Processing SSP245_G6, Ensemble 01
Processing SSP245_G6, Ensemble 02
Processing SSP245_G6, Ensemble 03
Saving global sum in /glade/work/awells/air_quality/CESM/mortality/ozone/Mortality_global_sum_CESM2_SSP245_G6_2020-2083.nc
